# TP 1 — Mon premier Perceptron
**Cours IA & Cybersécurité — Master 1**

---

## Objectifs
À la fin de ce TP, vous serez capable de :
1. Expliquer ce qu'est un perceptron et comment il apprend
2. Implémenter un perceptron **from scratch** en NumPy
3. Visualiser la frontière de décision apprise
4. Reproduire le même résultat avec **PyTorch** en 10 lignes

**Durée estimée** : 2h  
**Prérequis** : Python de base, aucune connaissance en ML requise

---

## 0. Imports et configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Graine aléatoire pour la reproductibilité
np.random.seed(42)

print("Librairies importées avec succès !")

---
## Partie 1 — Comprendre le perceptron

Un **perceptron** est le neurone artificiel le plus simple. Il prend des entrées, les multiplie par des **poids**, additionne tout, et décide : 0 ou 1 ?

```
x1 ──(w1)──┐
            ├──[ somme pondérée ]──[ activation ]──▶ ŷ (0 ou 1)
x2 ──(w2)──┘
             + biais (b)
```

**Formule** : `ŷ = 1 si (w1*x1 + w2*x2 + b) > 0, sinon 0`

**Apprentissage** : si le perceptron se trompe, on ajuste les poids dans la bonne direction (règle de Hebb).

---
## Partie 2 — Jeu de données : trafic réseau (simplifié)

On simule un problème de cybersécurité : **classifier du trafic réseau** en normal (0) ou suspect (1).

Nos deux features :
- **x1** = nombre de connexions par seconde (normalisé)
- **x2** = taille moyenne des paquets (normalisée)

In [ ]:
# Génération du dataset
n = 100  # 100 exemples par classe

# Trafic NORMAL : peu de connexions, paquets de taille classique
X_normal = np.random.randn(n, 2) * 0.6 + np.array([-1.5, -1.0])

# Trafic SUSPECT : beaucoup de connexions, petits paquets (scan de port ?)
X_suspect = np.random.randn(n, 2) * 0.6 + np.array([1.5, 1.0])

# Assemblage
X = np.vstack([X_normal, X_suspect])   # shape (200, 2)
y = np.array([0]*n + [1]*n)            # 0 = normal, 1 = suspect

print(f"Dataset : {X.shape[0]} exemples, {X.shape[1]} features")
print(f"Classes : {np.sum(y==0)} normaux, {np.sum(y==1)} suspects")

# Visualisation
plt.figure(figsize=(7, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', alpha=0.7, label='Normal', edgecolors='white', s=60)
plt.scatter(X[y==1, 0], X[y==1, 1], c='tomato',    alpha=0.7, label='Suspect', edgecolors='white', s=60)
plt.xlabel('Connexions / seconde (normalisé)')
plt.ylabel('Taille des paquets (normalisé)')
plt.title('Trafic réseau — dataset de classification')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Partie 3 — Perceptron from scratch (NumPy)

On implémente l'algorithme du perceptron étape par étape.

In [ ]:
class Perceptron:
    """
    Perceptron binaire — implémentation from scratch.
    
    Paramètres
    ----------
    lr    : taux d'apprentissage (learning rate)
    epochs: nombre de passes sur le dataset
    """
    
    def __init__(self, lr=0.1, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.errors_history = []  # pour tracer la courbe d'apprentissage
    
    def _activation(self, z):
        """Fonction échelon : 1 si z > 0, sinon 0"""
        return np.where(z > 0, 1, 0)
    
    def predict(self, X):
        """Calcule la prédiction pour un batch d'exemples."""
        z = X @ self.weights + self.bias   # combinaison linéaire
        return self._activation(z)         # seuillage
    
    def fit(self, X, y):
        """
        Entraîne le perceptron sur (X, y).
        
        Règle de mise à jour :
            erreur = y_reel - y_predit
            w ← w + lr * erreur * x
            b ← b + lr * erreur
        """
        n_samples, n_features = X.shape
        
        # Initialisation aléatoire des poids
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        for epoch in range(self.epochs):
            n_errors = 0
            
            # On passe sur chaque exemple un par un
            for xi, yi in zip(X, y):
                y_hat = self._activation(np.dot(xi, self.weights) + self.bias)
                erreur = yi - y_hat
                
                # Mise à jour des poids
                self.weights += self.lr * erreur * xi
                self.bias    += self.lr * erreur
                
                if erreur != 0:
                    n_errors += 1
            
            self.errors_history.append(n_errors)
            
            # Affichage toutes les 10 époques
            if (epoch + 1) % 10 == 0:
                acc = 1 - n_errors / n_samples
                print(f"Époque {epoch+1:3d}/{self.epochs} | Erreurs : {n_errors:3d} | Accuracy : {acc:.1%}")
        
        return self

In [ ]:
# Entraînement
perceptron = Perceptron(lr=0.1, epochs=50)
perceptron.fit(X, y)

### Visualisation de la frontière de décision

In [ ]:
def plot_decision_boundary(model, X, y, title="Frontière de décision"):
    """Trace la frontière de décision du modèle sur les données."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    # Grille de points
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ── Graphique 1 : frontière de décision
    ax = axes[0]
    ax.contourf(xx, yy, Z, alpha=0.2, colors=['steelblue', 'tomato'])
    ax.contour(xx, yy, Z, colors='black', linewidths=1.5)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', alpha=0.8, edgecolors='white', s=60, label='Normal')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='tomato',    alpha=0.8, edgecolors='white', s=60, label='Suspect')
    ax.set_xlabel('Connexions / seconde')
    ax.set_ylabel('Taille des paquets')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    
    # ── Graphique 2 : courbe d'apprentissage
    ax2 = axes[1]
    ax2.plot(model.errors_history, color='slateblue', linewidth=2)
    ax2.fill_between(range(len(model.errors_history)), model.errors_history, alpha=0.15, color='slateblue')
    ax2.set_xlabel('Époque')
    ax2.set_ylabel('Nombre d\'erreurs')
    ax2.set_title('Courbe d\'apprentissage')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_decision_boundary(perceptron, X, y, "Perceptron — frontière de décision")

# Accuracy finale
y_pred = perceptron.predict(X)
accuracy = np.mean(y_pred == y)
print(f"\nAccuracy finale : {accuracy:.1%}")
print(f"Poids appris    : w = {perceptron.weights}, b = {perceptron.bias:.4f}")

---
## Partie 4 — Exercices

> **Répondez dans les cellules ci-dessous.**

### Exercice 4.1 — Influence du taux d'apprentissage

Entraînez 3 perceptrons avec `lr = 0.001`, `lr = 0.1` et `lr = 1.0`.  
Tracez leurs courbes d'apprentissage sur le même graphique.  
Que constatez-vous ? Lequel converge le plus vite ? Le mieux ?

In [ ]:
# Votre code ici
learning_rates = [0.001, 0.1, 1.0]
couleurs = ['steelblue', 'slateblue', 'tomato']

plt.figure(figsize=(8, 5))

for lr, couleur in zip(learning_rates, couleurs):
    p = Perceptron(lr=lr, epochs=50)
    p.fit(X, y)
    plt.plot(p.errors_history, label=f'lr={lr}', color=couleur, linewidth=2)

plt.xlabel('Époque')
plt.ylabel('Erreurs')
plt.title('Impact du taux d\'apprentissage')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# TODO : répondez ici en commentaire
# Vos observations :

### Exercice 4.2 — Limite du perceptron (XOR)

Le perceptron ne peut apprendre que des frontières **linéaires**.  
Essayez de lui apprendre le XOR (problème non linéairement séparable).  
Qu'observez-vous ? Pourquoi ?

In [ ]:
# Dataset XOR
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])  # XOR

p_xor = Perceptron(lr=0.1, epochs=100)
p_xor.fit(X_xor, y_xor)

print("\nPrédictions sur XOR :")
for xi, yi in zip(X_xor, y_xor):
    pred = p_xor.predict(xi.reshape(1,-1))[0]
    status = "OK" if pred == yi else "ERREUR"
    print(f"  {xi} → attendu {yi}, prédit {pred}  [{status}]")

# TODO : expliquez pourquoi en commentaire

---
## Partie 5 — Même perceptron avec PyTorch

Maintenant qu'on comprend les mécanismes, voyons comment PyTorch fait la même chose en quelques lignes — et peut aller bien plus loin (GPU, réseaux profonds, etc.).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print(f"PyTorch version : {torch.__version__}")

# Conversion en tenseurs PyTorch
X_torch = torch.FloatTensor(X)
y_torch = torch.FloatTensor(y).unsqueeze(1)  # shape (200, 1)

# ── Définition du modèle (1 couche, 1 neurone) ──────────────────────────
class PerceptronPyTorch(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(in_features=2, out_features=1)
    
    def forward(self, x):
        return torch.sigmoid(self.fc(x))  # sigmoid : sortie entre 0 et 1

model = PerceptronPyTorch()

# ── Fonction de coût et optimiseur ──────────────────────────────────────
criterion = nn.BCELoss()                           # Binary Cross Entropy
optimizer = optim.SGD(model.parameters(), lr=0.1)  # Descente de gradient

# ── Boucle d'entraînement ───────────────────────────────────────────────
losses = []
for epoch in range(100):
    optimizer.zero_grad()          # 1. Réinitialise les gradients
    y_hat = model(X_torch)         # 2. Forward pass
    loss = criterion(y_hat, y_torch)  # 3. Calcul de la perte
    loss.backward()                # 4. Backpropagation
    optimizer.step()               # 5. Mise à jour des poids
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Époque {epoch+1:3d} | Loss : {loss.item():.4f}")

print("\nEntraînement terminé !")

In [ ]:
# Évaluation du modèle PyTorch
with torch.no_grad():
    y_prob = model(X_torch).numpy().flatten()
    y_pred_pt = (y_prob > 0.5).astype(int)

accuracy_pt = np.mean(y_pred_pt == y)
print(f"Accuracy PyTorch : {accuracy_pt:.1%}")

# Courbe de loss
plt.figure(figsize=(7, 4))
plt.plot(losses, color='slateblue', linewidth=2)
plt.fill_between(range(len(losses)), losses, alpha=0.15, color='slateblue')
plt.xlabel('Époque')
plt.ylabel('Loss (BCE)')
plt.title('Courbe de loss — Perceptron PyTorch')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Poids appris
w = model.fc.weight.data.numpy()[0]
b = model.fc.bias.data.item()
print(f"\nPoids appris : w = {w}, b = {b:.4f}")

---
## Partie 6 — Comparaison et synthèse

Complétez le tableau ci-dessous en exécutant les cellules précédentes.

In [ ]:
print("="*55)
print(f"{'':30s} {'NumPy':12s} {'PyTorch':12s}")
print("-"*55)
print(f"{'Accuracy finale':30s} {np.mean(perceptron.predict(X) == y):.1%}        {accuracy_pt:.1%}")
print(f"{'Poids w1':30s} {perceptron.weights[0]:+.4f}       {w[0]:+.4f}")
print(f"{'Poids w2':30s} {perceptron.weights[1]:+.4f}       {w[1]:+.4f}")
print(f"{'Biais b':30s} {perceptron.bias:+.4f}       {b:+.4f}")
print("="*55)
print("\nLes deux approches convergent vers les mêmes poids !")

---
## Bilan et questions de réflexion

Répondez brièvement à ces questions dans la cellule ci-dessous :

1. Pourquoi le perceptron échoue-t-il sur XOR ?
2. Quelle est la différence entre la règle du perceptron (NumPy) et la descente de gradient stochastique (PyTorch) ?
3. Dans un contexte cybersécurité, pourquoi est-ce dangereux de faire confiance à 100% à un modèle ayant 95% d'accuracy ?

**Vos réponses :**

1. ...
2. ...
3. ...

---
## Rendu

Avant de remettre ce notebook :
- [ ] Toutes les cellules de code sont exécutées (`Kernel > Restart & Run All`)
- [ ] Les exercices 4.1 et 4.2 sont complétés avec vos observations
- [ ] Les 3 questions de réflexion ont une réponse
- [ ] Le fichier est sauvegardé (`Ctrl+S`)

**Nommez votre fichier** : `tp1_NOM_Prenom.ipynb` et déposez-le sur le portail du cours.